In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
import os
import json 
import pathlib
import requests



In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize with specific local optimizations
llm = ChatOllama(
    model="llama3.2:1b",
    tools=[],
    temperature=0.3, # Lower temperature for more factual consistency
)

# 2. Use a simple Chain instead of an Agent (1B models handle chains much better)
prompt = ChatPromptTemplate.from_template("""
Answer the following question as a helpful assistant.
Question: {input}
Answer:""")


chain = prompt | llm | StrOutputParser()

# 3. Execution

# response = chain.stream({"input": "why do parrots talk?"})
# for chunk in response:
#     print(chunk, end="", flush=True)

In [94]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel ,Field

from dotenv import load_dotenv

load_dotenv()

# 1. Define your Schema
class ProductInfo(BaseModel):
    name: str= Field(..., description="The name of the product")
    price: float    = Field(..., description="The price of the product in USD , usd Must be written in small letters")     
    category: str   = Field(..., description="The category of the product")
    in_stock: bool  = Field(..., description="Whether the product is currently in stock")   

client = ChatOllama(
    model="llama3.2:1b",
    temperature=0.7

)
structured_llm = client.with_structured_output(ProductInfo)
response = structured_llm.invoke(
    "What is the price and availability of the latest MacBook Pro?"
    )


In [95]:
response.model_dump_json()


'{"name":"MacBook Pro","price":1299.0,"category":"Laptops","in_stock":true}'

In [93]:
from langchain_ollama import ChatOllama # Use ChatOllama for structured support
from pydantic import BaseModel , Field
from dotenv import load_dotenv

load_dotenv()

# 1. Define your Schema
class ProductInfo(BaseModel):
    name: str= Field(..., description="The name of the product")
    price: float    = Field(..., description="The price of the product in USD")     
    category: str   = Field(..., description="The category of the product")
    in_stock: bool  = Field(..., description="Whether the product is currently in stock")   

# 2. Initialize the Chat model
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0  # Set to 0 for more reliable structured data
)

# 3. Bind the Pydantic model to the LLM
structured_llm = llm.with_structured_output(ProductInfo)

# 4. Invoke and get a REAL Python object
product = structured_llm.invoke(
    "What is the price and availability of the latest MacBook Pro 14 pro?"
)

# Now these will work!
print(f"Product: {product.name}")
print(f"Price: ${product.price}")
print(f"In Stock: {product.in_stock}")

Product: MacBook Pro 14
Price: $1799.0
In Stock: True


In [5]:
for chunk in chain.stream({"input": "why do parrots talk?"}):
    print(chunk, end="")

Parrots are known for their ability to mimic human speech and other sounds, and they do it for several reasons. Here are some possible explanations:

1. **Communication and social interaction**: Parrots are social birds that live in flocks, and they use vocalizations to communicate with each other. By mimicking human speech, parrots can initiate conversations, express themselves, and even learn from their human caregivers.
2. **Learning and problem-solving**: Parrots are intelligent birds that can learn from their environment and adapt to new situations. By mimicking human speech, they may be trying to learn new words, phrases, or even solve problems.
3. **Attention and interaction**: Parrots are known to be highly attentive and responsive to their human caregivers. By mimicking human speech, they may be trying to get attention, affection, or even playtime.
4. **Evolutionary advantage**: In the wild, parrots have evolved to mimic other sounds, such as bird calls, to communicate with ot

In [10]:
response = chain.batch(["What is the capital of France?", "What is the name of largest and most intelligent mammal?"])
for res in response:
    print(res)

The capital of France is Paris.
The largest and most intelligent mammal is the blue whale (Balaenoptera musculus). On average, an adult blue whale can weigh around 150-170 tons (136,000-152,000 kg) and reach lengths of up to 82 feet (25 meters). However, the largest blue whale ever recorded was a female that was found in 1947 off the coast of Iceland, which weighed around 210 tons (182,000 kg) and measured around 108 feet (33 meters) in length.

As for intelligence, blue whales are considered one of the most intelligent animals on Earth. They have been observed using complex behaviors such as hunting, socializing, and even communicating with each other using low-frequency sounds. They have also been known to exhibit self-awareness and have been observed recognizing individual members of their own species.

It's worth noting that while blue whales are the largest and most intelligent mammals, they are not the only intelligent mammals. Other examples of intelligent mammals include elepha

In [27]:
from pydantic import BaseModel , Field 

class MyOutput(BaseModel):
    answer: str
chain = prompt | llm | StrOutputParser(output_cls=MyOutput)
chain.invoke({"input": "what is 2 + 2?"})

'2 + 2 = 4.'

In [98]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

# --- 1. Define your Models (Unchanged) ---
class LineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)

    @property
    def total(self) -> float:
        return self.quantity * self.unit_price

class Invoice(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    vendor_address: str | None = None
    items: list[LineItem]
    subtotal: float
    tax: float | None = None
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"

# --- 2. Initialize ChatOllama ---
# Use temperature 0 and ensure you use ChatOllama, not OllamaLLM
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

# --- 3. Bind the Schema ---
# method="json_schema" is the most reliable for local models
structured_llm = llm.with_structured_output(Invoice, method="json_schema")

# --- 4. Invoke ---
invoice_text = """
Invoice #INV-2025-001
Date: January 15, 2025
From: Acme Corp, 123 Business St

Items:
- Widget Pro (5) @ $29.99 each
- Service Fee (1) @ $50.00

Subtotal: $199.95
Tax: $16.00
Total: $215.95

Status: Paid
"""

# Invoke returns a validated Invoice object directly
invoice = structured_llm.invoke(f"Extract all data from this invoice text accurately: \n{invoice_text}")

# --- 5. Print Results ---
print(f"Invoice Number: {invoice.invoice_number}")
print(f"Vendor: {invoice.vendor_name}")
print(f"Date: {invoice.date}")
print(f"Vendor Address: {invoice.vendor_address or 'N/A'}") 
print(f"Subtotal: ${invoice.subtotal:.2f}")
print(f"Tax: ${invoice.tax or 0:.2f}")
print(f"Total: ${invoice.total:.2f}")
print(f"Status: {invoice.payment_status}")
for item in invoice.items:
    print(f"  - {item.description}: {item.quantity} x ${item.unit_price:.2f} (Total: ${item.total:.2f})")

Invoice Number: INV-2025-001
Vendor: Acme Corp
Date: January 15, 2025
Vendor Address: N/A
Subtotal: $199.95
Tax: $16.00
Total: $215.95
Status: paid
  - Widget Pro (5): 5 x $29.99 (Total: $149.95)
  - Service Fee (1): 1 x $50.00 (Total: $50.00)


In [ ]:
from pydantic import BaseModel , Field 

class MyOutput(BaseModel):
    answer: str
chain = prompt | llm | StrOutputParser(output_cls=MyOutput)
chain.invoke({"input": "what is 2 + 2?"})

'2 + 2 = 4.'

In [99]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import Literal, List
from dotenv import load_dotenv

load_dotenv()

# --- 1. Define your Schema (Keep your existing LineItem and Invoice classes) ---
class LineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)

class Invoice(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    items: List[LineItem]
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"

# --- 2. Initialize the Structured LLM ---
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0,
    format="json"
)
# Using json_schema is CRITICAL for the 1B model to handle lists of items
structured_llm = llm.with_structured_output(Invoice, method="json_schema")

# --- 3. List of Invoices to Process ---
invoices_data = [
    "Invoice #INV-001, Date: Jan 10, From: Apple Store. Item: iPhone 15 (1) @ $799. Total: $799. Status: Paid",
    "Invoice #INV-002, Date: Jan 12, From: Amazon. Item: USB-C Cable (2) @ $15. Total: $30. Status: Pending",
    "Invoice #6789, Date: Feb 01, From: Cloud Hosting. Item: Server Pro (1) @ $50. Total: $50. Status: Overdue",
    "Invoice #TX-99, Date: Feb 05, From: Local Cafe. Item: Coffee (10) @ $4.50. Total: $45. Status: Paid"
]

# --- 4. Execution Loop ---
processed_invoices = []

print(f"--- Starting Extraction for {len(invoices_data)} invoices ---")

for i, text in enumerate(invoices_data, 1):
    try:
        print(f"Processing Invoice {i}...")
        # invoke returns a validated Invoice object
        result = structured_llm.invoke(f"Extract structured data from this invoice text: {text}")
        processed_invoices.append(result)
    except Exception as e:
        print(f"Error processing invoice {i}: {e}")

# --- 5. Output Results ---
print("\n--- Final Results ---")
for inv in processed_invoices:
    print(f"[{inv.invoice_number}] {inv.vendor_name} - ${inv.total} ({inv.payment_status})")

--- Starting Extraction for 4 invoices ---
Processing Invoice 1...
Processing Invoice 2...
Processing Invoice 3...
Processing Invoice 4...

--- Final Results ---
[INV-001] Apple Store - $799.0 (paid)
[INV-002] Amazon - $30.0 (pending)
[6789] Cloud Hosting - $50.0 (overdue)
[TX-99] Local Cafe - $45.0 (paid)


## TO process an Image 


In [103]:
import base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# 1. Helper to encode image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 2. Setup model (Use a vision model!)
llm = ChatOllama(model="moondream")

# 3. Create message with image
image_b64 = encode_image("C:/Users/hp/Desktop/download.jpg")

message = HumanMessage(
    content=[
        {"type": "text", "text": "what is in this image?"},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
        },
    ]
)

# 4. Invoke
response = llm.invoke([message])
print(response.content)


The image features a small, adorable pig with a bow on its head. The pig is sitting in a field of leaves, giving the impression that it is in a forest or a park. The pig is looking directly at the camera, making it seem as if it is posing for a photo. The background of the image is blurred, drawing attention to the pig and its bow.


In [107]:
import base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# 1. Path to your file (using 'r' for Windows backslashes)
IMAGE_PATH = r"C:\Users\hp\Desktop\465781305-8751e031-61e8-4ef2-9823-5e4316bd6356.gif"

# 2. Helper function to convert image to Base64
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 3. Setup the Vision Model
# I recommend 'moondream' for your i5 12th gen for near-instant results
llm = ChatOllama(model="moondream")

# 4. Create the message payload
image_base64 = encode_image(IMAGE_PATH)

message = HumanMessage(
    content=[
        {"type": "text", "text": " According to what you see can you explain What is happening in this image Step by step?"},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{image_base64}"},
        },
    ]
)

# 5. Get the response
print("Sending image to Ollama...")
response = llm.invoke([message])

print("\n--- Model Analysis ---")
print(response.content)


Sending image to Ollama...

--- Model Analysis ---

The image presents a detailed flowchart of a computer system, specifically focusing on the concept of "Multiple Transformer Blocks". The flowchart is color-coded, with each color representing a different component or process within the system. The flowchart is organized into multiple steps, providing a clear and comprehensive view of the system's functionality. The flowchart is set against a white background, making it easy to read and understand.


In [5]:
import os
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

# --- 1. Schemas (Unchanged) ---
class MovieRef(BaseModel):
    title: str = Field(..., description="Official movie title")
    year: Optional[int] = Field(None, description="Release year if known")

class MoviePick(BaseModel):
    movie: MovieRef
    why_known: str = Field(..., description="Short reason why this movie is well-known")
    confidence: float = Field(..., ge=0, le=1, description="Confidence from 0 to 1")

class PersonRole(BaseModel):
    name: str
    role: str = Field(..., description="Character or production role")

class MovieMeta(BaseModel):
    genres: List[str]
    language: Optional[str] = None
    country: Optional[str] = None
    runtime_minutes: Optional[int] = None

class MovieRatings(BaseModel):
    imdb: Optional[float] = Field(None, ge=0, le=10)
    rotten_tomatoes: Optional[int] = Field(None, ge=0, le=100)

class MovieDetails(BaseModel):
    movie: MovieRef
    plot: str
    themes: List[str]
    key_people: List[PersonRole]
    metadata: MovieMeta
    ratings: MovieRatings

# --- 2. Initialize Gemini Model ---
# We use gemini-3.1-flash-lite-preview or gemini-2.0-flash (best for speed/structured tasks)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

# --- 3. Step 1: Pick a movie ---
# Gemini handles structured output natively via its API
pick_llm = llm.with_structured_output(MoviePick)
movie_pick = pick_llm.invoke(
    "Choose ONE real, famous movie you are very confident you know well. "
    "Return valid structured data only."
)

# --- 4. Step 2: Ask about the same movie ---
details_llm = llm.with_structured_output(MovieDetails)
movie_details = details_llm.invoke(
    f"Provide accurate structured details for this movie: "
    f"{movie_pick.movie.title} ({movie_pick.movie.year})."
)

# --- 5. Print Results ---
print(f"Picked movie: {movie_pick.movie.title} ({movie_pick.movie.year})")
print("\nMoviePick JSON:")
print(movie_pick.model_dump_json(indent=2))

print("\nMovieDetails JSON:")
print(movie_details.model_dump_json(indent=2))


Picked movie: The Shawshank Redemption (1994)

MoviePick JSON:
{
  "movie": {
    "title": "The Shawshank Redemption",
    "year": 1994
  },
  "why_known": "Widely considered one of the greatest films of all time, it holds the top spot on the IMDb Top 250 list.",
  "confidence": 1.0
}

MovieDetails JSON:
{
  "movie": {
    "title": "The Shawshank Redemption",
    "year": 1994
  },
  "plot": "A banker convicted of murdering his wife and her lover maintains his innocence over two decades in Shawshank State Penitentiary, where he forms a deep friendship with a fellow inmate and finds ways to maintain his hope and dignity.",
  "themes": [
    "Hope",
    "Friendship",
    "Redemption",
    "Institutionalization",
    "Justice"
  ],
  "key_people": [
    {
      "name": "Tim Robbins",
      "role": "Andy Dufresne"
    },
    {
      "name": "Morgan Freeman",
      "role": "Ellis Boyd 'Red' Redding"
    },
    {
      "name": "Frank Darabont",
      "role": "Director"
    }
  ],
  "metadata"